In [1]:
from pathlib import Path
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score


In [3]:
from pathlib import Path

# Notebook is inside PharmaLink/notebooks
BASE_DIR = Path.cwd().parent   # <-- THIS IS THE FIX

def resolve_dirs(base_dir: Path):
    candidates = [
        (base_dir / "backend" / "data", base_dir / "artifacts"),  # your real structure
        (base_dir / "data", base_dir / "model"),                  # alternate structure
    ]
    for d, m in candidates:
        if d.exists() and m.exists():
            return d, m

    raise FileNotFoundError(
        f"Could not find data/model directories under {base_dir}"
    )

DATA_DIR, MODEL_DIR = resolve_dirs(BASE_DIR)

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR, "exists:", DATA_DIR.exists())
print("MODEL_DIR:", MODEL_DIR, "exists:", MODEL_DIR.exists())


BASE_DIR: C:\Users\User\OneDrive - Sri Lanka Institute of Information Technology\Desktop\Research\PharmaLink
DATA_DIR: C:\Users\User\OneDrive - Sri Lanka Institute of Information Technology\Desktop\Research\PharmaLink\data exists: True
MODEL_DIR: C:\Users\User\OneDrive - Sri Lanka Institute of Information Technology\Desktop\Research\PharmaLink\model exists: True


In [4]:
food_subset_path = DATA_DIR / "food_subset.csv"
new_foodset_path = DATA_DIR / "new_foodset.csv"

food_subset = pd.read_csv(food_subset_path)
new_foodset = pd.read_csv(new_foodset_path)

print("food_subset shape:", food_subset.shape)
print("new_foodset shape:", new_foodset.shape)

food_subset.head()


food_subset shape: (766, 15)
new_foodset shape: (855, 13)


,Food,energy,protein,fat,carbs,fiber,calcium,iron,vitamin_c,folate,vitamin_a,vitamin_e,is_alcohol,is_leafy_green,vitamin_k_proxy
0,Beer,43.0,0.0,0.46,3.55,0.0,4.0,0.02,0.0,6.0,0.0,0.0,1,0,0
1,Arak,222.0,0.0,0.00,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,1,0,0
2,Kasippu,222.0,0.0,0.00,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,1,0,0
3,"LIQUOR, TODDY, COCONUT",222.0,0.0,0.00,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,1,0,0
4,"LIQUOR, TODDY, KITUL",222.0,0.0,0.00,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,1,0,0


In [5]:
ALLOWED_TYPES = {"main","curry","vegetable","veg","protein","dessert","drink","side","unknown"}

def normalize_text(x):
    return str(x).strip().lower()

def normalize_food_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Ensure Food column
    if "Food" not in df.columns:
        if "Food Product" in df.columns:
            df["Food"] = df["Food Product"]
        elif "Food_Item" in df.columns:
            df["Food"] = df["Food_Item"]
        else:
            raise ValueError("No Food column found in this CSV.")

    # Ensure food_type column
    if "food_type" not in df.columns:
        if "Food_type" in df.columns:
            df["food_type"] = df["Food_type"]
        elif "Food Type" in df.columns:
            df["food_type"] = df["Food Type"]
        else:
            df["food_type"] = "unknown"

    df["Food"] = df["Food"].astype(str).map(normalize_text)
    df["food_type"] = df["food_type"].astype(str).map(normalize_text)

    # Clean labels
    df.loc[~df["food_type"].isin(ALLOWED_TYPES), "food_type"] = "unknown"
    df.loc[df["food_type"] == "veg", "food_type"] = "vegetable"

    df = df[df["Food"].notna() & (df["Food"] != "")]
    df = df.drop_duplicates(subset=["Food"], keep="first")
    return df[["Food", "food_type"]]

df1 = normalize_food_df(food_subset)
df2 = normalize_food_df(new_foodset)

all_foods = pd.concat([df1, df2], ignore_index=True).drop_duplicates(subset=["Food"], keep="first")
train_df = all_foods[all_foods["food_type"] != "unknown"].copy()

print("All foods:", len(all_foods))
print("Labeled foods for training:", len(train_df))
train_df["food_type"].value_counts()


All foods: 1517
Labeled foods for training: 752


food_type
side         464
main         253
vegetable     13
protein        8
curry          7
drink          4
dessert        3
Name: count, dtype: int64

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Merge rare classes
min_count = 10
vc = train_df["food_type"].value_counts()
rare = vc[vc < min_count].index.tolist()
print("Rare classes:", rare)

train_df2 = train_df.copy()
train_df2.loc[train_df2["food_type"].isin(rare), "food_type"] = "side"

# ✅ USE train_df2 (not train_df)
X = train_df2["Food"].tolist()
y = train_df2["food_type"].tolist()

# stratify is safe now because rare classes removed/merged
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = Pipeline(
    steps=[
        ("tfidf", TfidfVectorizer(ngram_range=(1, 3), min_df=1)),
        ("clf", LogisticRegression(max_iter=4000, class_weight="balanced")),
    ]
)

model.fit(X_train, y_train)

preds = model.predict(X_test)
acc = accuracy_score(y_test, preds)

print("Accuracy:", round(acc, 4))
print(classification_report(y_test, preds, zero_division=0))


Rare classes: ['protein', 'curry', 'drink', 'dessert']
Accuracy: 0.8079
              precision    recall  f1-score   support

        main       0.76      0.73      0.74        51
        side       0.86      0.86      0.86        97
   vegetable       0.40      0.67      0.50         3

    accuracy                           0.81       151
   macro avg       0.67      0.75      0.70       151
weighted avg       0.81      0.81      0.81       151



In [9]:
import joblib

out_path = MODEL_DIR / "food_type_model.pkl"
joblib.dump(model, out_path)

print("Saved model to:", out_path)


Saved model to: C:\Users\User\OneDrive - Sri Lanka Institute of Information Technology\Desktop\Research\PharmaLink\model\food_type_model.pkl


In [10]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# If your notebook is PharmaLink/notebooks/
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"
MODEL_DIR = BASE_DIR / "model"
MODEL_DIR.mkdir(exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR exists:", DATA_DIR.exists(), DATA_DIR)
print("MODEL_DIR exists:", MODEL_DIR.exists(), MODEL_DIR)


BASE_DIR: C:\Users\User\OneDrive - Sri Lanka Institute of Information Technology\Desktop\Research\PharmaLink
DATA_DIR exists: True C:\Users\User\OneDrive - Sri Lanka Institute of Information Technology\Desktop\Research\PharmaLink\data
MODEL_DIR exists: True C:\Users\User\OneDrive - Sri Lanka Institute of Information Technology\Desktop\Research\PharmaLink\model


In [11]:
food_subset = pd.read_csv(DATA_DIR / "food_subset.csv")
meal_foods = pd.read_csv(DATA_DIR / "new_foodset.csv")

# Normalize column names like your API
def normalize_food_df(df):
    df = df.rename(columns={
        "Food_Item": "Food",
        "Food Product": "Food",
        "Calories": "energy",
        "Calories (kcal)": "energy",
        "Energy": "energy",
        "Protein (g)": "protein",
        "Carbohydrate (g)": "carbs",
        "Fat (g)": "fat",
        "Fiber (g)": "fiber",
        "Fibre (g)": "fiber",
        "Sugars (g)": "sugars",
        "Sugar (g)": "sugars",
        "Sodium (mg)": "sodium",
        "Meal_Type": "meal_type",
        "Food_type": "food_type",
        "Food Type": "food_type",
    })
    return df

food_subset = normalize_food_df(food_subset)
meal_foods = normalize_food_df(meal_foods)

core_cols = ["Food","energy","carbs","fiber","sugars","sodium","fat","protein"]
for c in core_cols:
    if c not in food_subset.columns: food_subset[c] = 0
    if c not in meal_foods.columns: meal_foods[c] = 0

foods = pd.concat([food_subset[core_cols], meal_foods[core_cols]], ignore_index=True)
foods["Food"] = foods["Food"].astype(str).str.strip()
foods = foods[foods["Food"] != ""].drop_duplicates(subset=["Food"]).reset_index(drop=True)

# numeric cleanup
for c in ["energy","carbs","fiber","sugars","sodium","fat","protein"]:
    foods[c] = pd.to_numeric(foods[c], errors="coerce").fillna(0.0)

foods.head()


,Food,energy,carbs,fiber,sugars,sodium,fat,protein
0,Beer,43.0,3.55,0.0,0.0,0.0,0.46,0.0
1,Arak,222.0,0.00,0.0,0.0,0.0,0.00,0.0
2,Kasippu,222.0,0.00,0.0,0.0,0.0,0.00,0.0
3,"LIQUOR, TODDY, COCONUT",222.0,0.00,0.0,0.0,0.0,0.00,0.0
4,"LIQUOR, TODDY, KITUL",222.0,0.00,0.0,0.0,0.0,0.00,0.0


In [12]:
MEAT_KEYWORDS = [
    "chicken","beef","pork","mutton","lamb","fish","tuna","salmon","shrimp",
    "bacon","sausage","ham","anchovy","bison","turkey","duck","crab","prawn","steak","burger"
]
HIGH_SUGAR_KEYWORDS = ["cake","cookie","soda","cola","candy","ice cream","chocolate","sweet","syrup"]
HIGH_SODIUM_KEYWORDS = ["pickle","canned","processed","instant","soy sauce","chips","noodles","salted"]

def normalize_name(x):
    return str(x).strip().lower()

def label_vegetarian(name: str) -> int:
    n = normalize_name(name)
    return 0 if any(k in n for k in MEAT_KEYWORDS) else 1

def label_diabetic(row) -> int:
    n = normalize_name(row["Food"])
    carbs = float(row["carbs"])
    fiber = float(row["fiber"])
    sugars = float(row["sugars"])

    if any(k in n for k in HIGH_SUGAR_KEYWORDS):
        return 0
    if sugars > 0:
        return 1 if sugars <= 15 else 0
    if carbs > 45 and fiber < 5:
        return 0
    return 1

def label_low_sodium(row) -> int:
    sodium = float(row["sodium"])
    if sodium > 0:
        return 1 if sodium <= 140 else 0
    n = normalize_name(row["Food"])
    return 0 if any(k in n for k in HIGH_SODIUM_KEYWORDS) else 1

foods["y_vegetarian"] = foods["Food"].apply(label_vegetarian)
foods["y_diabetic"] = foods.apply(label_diabetic, axis=1)
foods["y_low_sodium"] = foods.apply(label_low_sodium, axis=1)

foods[["Food","y_vegetarian","y_diabetic","y_low_sodium"]].head(20)


,Food,y_vegetarian,y_diabetic,y_low_sodium
0,Beer,1,1,1
1,Arak,1,1,1
2,Kasippu,1,1,1
3,"LIQUOR, TODDY, COCONUT",1,1,1
4,"LIQUOR, TODDY, KITUL",1,1,1
5,"LIQUOR, TODDY, PALMYRA",1,1,1
6,"LIQUOR, HARD (>35% ABV), NON-SPECIFIC",1,1,1
7,"Coconut water(botteled, sweetend)",1,0,1
8,"Coconut water(fresh, unsweetend)",1,0,1
9,Fruit and Milk Drink,1,1,1


In [14]:
MEAT_KEYWORDS = [
    "chicken","beef","pork","mutton","lamb","fish","tuna","salmon","shrimp",
    "bacon","sausage","ham","anchovy","bison","turkey","duck","crab","prawn","steak","burger"
]
HIGH_SUGAR_KEYWORDS = ["cake","cookie","soda","cola","candy","ice cream","chocolate","sweet","syrup"]
HIGH_SODIUM_KEYWORDS = ["pickle","canned","processed","instant","soy sauce","chips","noodles","salted"]

def normalize_name(x):
    return str(x).strip().lower()

def label_vegetarian(name: str) -> int:
    n = normalize_name(name)
    return 0 if any(k in n for k in MEAT_KEYWORDS) else 1

def label_diabetic(row) -> int:
    n = normalize_name(row["Food"])
    carbs = float(row["carbs"])
    fiber = float(row["fiber"])
    sugars = float(row["sugars"])

    if any(k in n for k in HIGH_SUGAR_KEYWORDS):
        return 0
    if sugars > 0:
        return 1 if sugars <= 15 else 0
    if carbs > 45 and fiber < 5:
        return 0
    return 1

def label_low_sodium(row) -> int:
    sodium = float(row["sodium"])
    if sodium > 0:
        return 1 if sodium <= 140 else 0
    n = normalize_name(row["Food"])
    return 0 if any(k in n for k in HIGH_SODIUM_KEYWORDS) else 1

foods["y_vegetarian"] = foods["Food"].apply(label_vegetarian)
foods["y_diabetic"] = foods.apply(label_diabetic, axis=1)
foods["y_low_sodium"] = foods.apply(label_low_sodium, axis=1)

foods[["Food","y_vegetarian","y_diabetic","y_low_sodium"]].head(20)


,Food,y_vegetarian,y_diabetic,y_low_sodium
0,Beer,1,1,1
1,Arak,1,1,1
2,Kasippu,1,1,1
3,"LIQUOR, TODDY, COCONUT",1,1,1
4,"LIQUOR, TODDY, KITUL",1,1,1
5,"LIQUOR, TODDY, PALMYRA",1,1,1
6,"LIQUOR, HARD (>35% ABV), NON-SPECIFIC",1,1,1
7,"Coconut water(botteled, sweetend)",1,0,1
8,"Coconut water(fresh, unsweetend)",1,0,1
9,Fruit and Milk Drink,1,1,1


In [15]:
foods.to_csv(DATA_DIR / "food_preferences_labeled.csv", index=False)
print("Saved:", DATA_DIR / "food_preferences_labeled.csv")


Saved: C:\Users\User\OneDrive - Sri Lanka Institute of Information Technology\Desktop\Research\PharmaLink\data\food_preferences_labeled.csv


In [16]:
FEATURE_NUM = ["energy","carbs","fiber","sugars","sodium","fat","protein"]

def train_binary_model(df: pd.DataFrame, target_col: str, model_name: str):
    df = df.copy()
    df["Food"] = df["Food"].astype(str)

    X = df[["Food"] + FEATURE_NUM]
    y = df[target_col].astype(int)

    # ensure both classes exist
    if y.nunique() < 2:
        raise ValueError(f"{target_col} has only one class. Need 0 and 1 samples.")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    pre = ColumnTransformer(
        transformers=[
            ("text", TfidfVectorizer(ngram_range=(1, 2), min_df=1), "Food"),
            ("num", StandardScaler(), FEATURE_NUM),
        ],
        remainder="drop",
    )

    clf = LogisticRegression(max_iter=5000, class_weight="balanced")

    pipe = Pipeline([("pre", pre), ("clf", clf)])
    pipe.fit(X_train, y_train)

    preds = pipe.predict(X_test)
    acc = accuracy_score(y_test, preds)

    print("="*60)
    print(model_name, "| target =", target_col)
    print("Accuracy:", round(acc, 4))
    print(classification_report(y_test, preds, zero_division=0))

    out_path = MODEL_DIR / model_name
    joblib.dump(pipe, out_path)
    print("Saved:", out_path)

    return pipe


In [17]:
veg_model = train_binary_model(foods, "y_vegetarian", "vegetarian_model.pkl")
dia_model = train_binary_model(foods, "y_diabetic", "diabetic_model.pkl")
low_model = train_binary_model(foods, "y_low_sodium", "low_sodium_model.pkl")


vegetarian_model.pkl | target = y_vegetarian
Accuracy: 0.9279
              precision    recall  f1-score   support

           0       0.69      0.93      0.79        45
           1       0.99      0.93      0.96       260

    accuracy                           0.93       305
   macro avg       0.84      0.93      0.87       305
weighted avg       0.94      0.93      0.93       305

Saved: C:\Users\User\OneDrive - Sri Lanka Institute of Information Technology\Desktop\Research\PharmaLink\model\vegetarian_model.pkl
diabetic_model.pkl | target = y_diabetic
Accuracy: 0.9705
              precision    recall  f1-score   support

           0       0.83      0.97      0.89        39
           1       1.00      0.97      0.98       266

    accuracy                           0.97       305
   macro avg       0.91      0.97      0.94       305
weighted avg       0.97      0.97      0.97       305

Saved: C:\Users\User\OneDrive - Sri Lanka Institute of Information Technology\Desktop\Researc

In [22]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().parent   # go from notebooks/ → PharmaLink/
DATA_DIR = BASE_DIR / "data"

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR exists:", DATA_DIR.exists())

df = pd.read_csv(DATA_DIR / "new_foodset.csv")



BASE_DIR: C:\Users\User\OneDrive - Sri Lanka Institute of Information Technology\Desktop\Research\PharmaLink
DATA_DIR exists: True


In [23]:
import os
print(os.listdir(DATA_DIR))


['drug_clean.csv', 'food_drug_pairs_gold.csv', 'food_drug_pairs_silver.csv', 'food_drug_pairs_train.csv', 'food_ingredients_and_allergens.csv', 'food_preferences_labeled.csv', 'food_subset.csv', 'new_foodset.csv']


In [25]:
print(df.columns.tolist())
df.head(3)


['Food_Item', 'Category', 'Calories (kcal)', 'Protein (g)', 'Carbohydrates (g)', 'Fat (g)', 'Fiber (g)', 'Sugars (g)', 'Sodium (mg)', 'Cholesterol (mg)', 'Meal_Type', 'Water_Intake (ml)', 'Food_type']


,Food_Item,Category,Calories (kcal),Protein (g),Carbohydrates (g),Fat (g),Fiber (g),Sugars (g),Sodium (mg),Cholesterol (mg),Meal_Type,Water_Intake (ml),Food_type
0,Scrambled Eggs (2 large),Protein/Dairy,180.0,12.0,2.0,14.0,0.0,1.0,180.0,370,Breakfast,250.0,main
1,Whole Wheat Toast (1 slice),Grain,80.0,4.0,14.0,1.0,2.0,2.0,140.0,0,Breakfast,0.0,side
2,Coffee (black),Beverage,5.0,0.3,0.0,0.1,0.0,0.0,5.0,0,Breakfast,0.0,side


In [26]:
import re

print("Columns:", df.columns.tolist())

# Find best column for food name
candidates = []
for c in df.columns:
    c_norm = re.sub(r"\s+", " ", str(c)).strip().lower()
    if "food" in c_norm and ("name" in c_norm or "item" in c_norm or "product" in c_norm or c_norm == "food"):
        candidates.append(c)

print("Detected food-name candidates:", candidates)

if not candidates:
    raise ValueError("❌ Could not find a food name column. Please tell me which column is food name.")
    
food_col = candidates[0]
print("✅ Using food column:", food_col)

df["Food"] = df[food_col].astype(str).str.strip()
df = df[df["Food"] != ""].drop_duplicates(subset=["Food"]).reset_index(drop=True)

df[["Food"]].head(10)


Columns: ['Food_Item', 'Category', 'Calories (kcal)', 'Protein (g)', 'Carbohydrates (g)', 'Fat (g)', 'Fiber (g)', 'Sugars (g)', 'Sodium (mg)', 'Cholesterol (mg)', 'Meal_Type', 'Water_Intake (ml)', 'Food_type']
Detected food-name candidates: ['Food_Item']
✅ Using food column: Food_Item


,Food
0,Scrambled Eggs (2 large)
1,Whole Wheat Toast (1 slice)
2,Coffee (black)
3,Banana
4,Grilled Chicken Salad
5,Apple
6,Salmon (4oz grilled)
7,Quinoa (1 cup cooked)
8,Steamed Broccoli (1 cup)
9,Greek Yogurt (plain 1 cup)


In [28]:
from pathlib import Path

Path("model").mkdir(exist_ok=True)


In [29]:
joblib.dump(cluster_pipe, "model/food_cluster_model.pkl")
print("✅ Saved model/food_cluster_model.pkl")


✅ Saved model/food_cluster_model.pkl


In [30]:
import os
print(os.listdir("model"))


['food_cluster_model.pkl']
